# Raw LLM vs. Mellea: Reliable Structured Output

Direct LLM calls can extract structured data, but they leave parsing, validation, and recovery to your application code. This notebook compares that raw approach with a Mellea workflow that returns typed objects and validates invoice math automatically. You will run both paths against the same invoice extraction task using a local Ollama model.

## What You'll Build

The use case is **invoice data extraction from unstructured email text** a common accounts-payable automation task where both approaches work, but reliability requirements determine which one belongs in production.

- A raw extraction flow that returns JSON text and parses it manually
- A Mellea extraction flow that returns typed Pydantic objects
- A reusable validator for invoice arithmetic
- A side-by-side comparison for clean and ambiguous inputs

### Prerequisites

- Python 3.11 or 3.12
- [Ollama](https://ollama.com/) installed locally
- A local chat model such as `granite4.1:8b`
- Basic familiarity with Pydantic

Pull the model before running the notebook:

```bash
ollama pull granite4.1:8b
```

### Setup

Install the dependencies with `uv`. This recipe uses [Ollama](https://ollama.com/).

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "mellea[litellm]" litellm pydantic
! echo "::endgroup::"

Import the libraries used in both implementations.

In [ ]:
import json
import re
from datetime import date
from typing import List, Optional

from litellm import completion
from mellea import start_session
from mellea.backends.model_options import ModelOption
from mellea.core.requirement import Requirement, ValidationResult
from pydantic import BaseModel, Field

Set the local model name once so both approaches use the same backend.

In [ ]:
OLLAMA_MODEL = "ollama/granite4.1:8b"

#### Step 1: Define the Data Model

Create Pydantic models for the invoice structure. Mellea will use these models as the target output format.

In [ ]:
class InvoiceItem(BaseModel):
    description: str = Field(description="Item or service description")
    quantity: int = Field(description="Quantity purchased")
    unit_price: float = Field(description="Price per unit")
    total: float = Field(description="Line item total")


class Invoice(BaseModel):
    invoice_number: str = Field(description="Invoice identifier")
    invoice_date: date = Field(description="Invoice date")
    vendor_name: str = Field(description="Vendor name")
    items: List[InvoiceItem] = Field(description="Purchased items")
    subtotal: float = Field(description="Subtotal before tax")
    tax: float = Field(description="Tax amount")
    total: float = Field(description="Final amount due")

#### Step 2: Prepare Sample Inputs

Use one clean invoice email and one ambiguous invoice email to compare behavior.

In [ ]:
sample_email = """
Subject: Invoice #INV-2024-0542 from TechSupplies Inc.

Invoice Number: INV-2024-0542
Date: January 15, 2024
Vendor: TechSupplies Inc.

Items:
1. Wireless Mouse - Quantity: 5 @ $25.00 each = $125.00
2. USB-C Cable (2m) - Quantity: 10 @ $12.50 each = $125.00
3. Laptop Stand - Quantity: 3 @ $45.00 each = $135.00

Subtotal: $385.00
Tax (8%): $30.80
Total: $415.80
"""

problematic_email = """
Subject: Invoice from TechSupplies

We sent you 5 wireless mice at twenty-five dollars each,
10 USB cables for $12.50 per cable,
and three laptop stands.

The total came to around $415 including tax.
"""

#### Step 3: Extract with a Raw LLM Call

Start with the direct approach. The model returns text, and your code must parse and validate the result manually.

In [ ]:
def _extract_json(text: str) -> str:
    match = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    return match.group(1) if match else text.strip()


def extract_invoice_raw(email_text: str) -> Optional[dict]:
    prompt = f"""
Extract invoice data from the email below.

Return ONLY valid JSON with this structure:
{{
  "invoice_number": "string",
  "invoice_date": "YYYY-MM-DD",
  "vendor_name": "string",
  "items": [
    {{
      "description": "string",
      "quantity": 0,
      "unit_price": 0.0,
      "total": 0.0
    }}
  ],
  "subtotal": 0.0,
  "tax": 0.0,
  "total": 0.0
}}

Email:
{email_text}
"""

    try:
        response = completion(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        response_text = _extract_json(response.choices[0].message.content)
        return json.loads(response_text)
    except json.JSONDecodeError as error:
        print(f"❌ JSON parsing failed: {error}")
        return None
    except Exception as error:
        print(f"❌ Raw extraction failed: {error}")
        return None

Run the raw path on the clean sample.

In [ ]:
print("RAW LLM APPROACH")

raw_result = extract_invoice_raw(sample_email)

if raw_result:
    print(f"\n✓ Got result of type: {type(raw_result)}")
    print(f"Invoice: {raw_result.get('invoice_number')}")
    print(f"Total: ${raw_result.get('total')}")
    print("\n⚠️ This is still an unvalidated dict.")
else:
    print("\n❌ Extraction failed")

#### Step 4: Extract with Mellea

Now create a Mellea session over the same Ollama model. Mellea will map the result into the `Invoice` schema.

The `start_session()` method initializes a connection to your LLM backend, creating a session object that persists across multiple interactions. The `instruct()` method sends instructions to the LLM with support for template variables (using `{{variable}}` syntax), structured output via the `format` parameter (pass a Pydantic model for type-safe results), and validation rules through the `requirements` parameter. This approach eliminates manual JSON parsing and provides automatic schema validation.

**Advanced: @generative Decorator**

For more complex scenarios, Mellea offers the `@generative` decorator, which converts function docstrings into prompts and type hints into schemas. This provides a more Pythonic way to define structured outputs compared to the `format` parameter, making your code more maintainable and self-documenting.

**ModelOption Enum**

The `ModelOption` enum (from `mellea.backends.ModelOption`) provides backend-agnostic inference options like `TEMPERATURE`, `SEED`, `MAX_NEW_TOKENS`, and `SYSTEM_PROMPT`. Using `ModelOption` keys ensures the same options work across all backends (Ollama, OpenAI, watsonx, etc.), making your code portable.

In [ ]:
m = start_session(
    backend_name="litellm",
    model_id=OLLAMA_MODEL,
)

In [ ]:
def extract_invoice_mellea(email_text: str) -> Invoice:
    result = m.instruct(
        "Extract invoice information from: {{email}}",
        user_variables={"email": email_text},
        format=Invoice,
        model_options={ModelOption.SEED: 42},
    )
    return Invoice.model_validate_json(result.value)

Run the Mellea path on the same sample.

In [ ]:
print("MELLEA APPROACH")

mellea_result = extract_invoice_mellea(sample_email)

print(f"\n✓ Got typed object: {type(mellea_result)}")
print(f"Invoice: {mellea_result.invoice_number}")
print(f"Date: {mellea_result.invoice_date}")
print(f"Vendor: {mellea_result.vendor_name}")
print(f"Total: ${mellea_result.total}")
print(f"Items: {len(mellea_result.items)}")

print("\nLine items:")
for item in mellea_result.items:
    print(f"  {item.description}: {item.quantity} × ${item.unit_price} = ${item.total}")

#### Step 5: Add Custom Validation

Add a business rule that checks invoice arithmetic. This catches outputs that are structurally valid but numerically wrong.

Mellea's requirements system enables automatic validation and repair through `Requirement` objects. When validation fails, Mellea can automatically retry the generation with the failure reason, allowing the LLM to correct its output. This instruct-validate-repair workflow ensures business rules are enforced without manual retry logic.

**Understanding check_only**

The `check_only=True` parameter in the `math_requirement` means the requirement description is evaluated after generation but NOT included in the prompt. This avoids the "purple elephant effect" — where mentioning something in a negative instruction (e.g., "do not mention purple elephants") paradoxically increases the chance the model produces it.

**Shorthand Constructors: req() and check()**

Mellea provides convenient shortcuts from `mellea.stdlib.requirements`:
- `req()` creates a standard Requirement (description included in prompt) — use for positive constraints
- `check()` creates a check-only Requirement (description NOT in prompt) — use for negative or hard-to-explain constraints

Example: `check("Do not mention purple elephants")` is better than including this instruction in the prompt.

In [ ]:
def validate_invoice_math(invoice: Invoice) -> bool:
    for item in invoice.items:
        if abs(item.total - (item.quantity * item.unit_price)) > 0.01:
            return False

    expected_subtotal = sum(item.total for item in invoice.items)
    if abs(invoice.subtotal - expected_subtotal) > 0.01:
        return False

    if abs(invoice.total - (invoice.subtotal + invoice.tax)) > 0.01:
        return False

    return True


def validate_invoice_requirement(context) -> ValidationResult:
    invoice = Invoice.model_validate_json(context.last_output().value)
    is_valid = validate_invoice_math(invoice)
    reason = "Invoice arithmetic is valid." if is_valid else "Invoice calculations are incorrect. Recalculate all totals."
    return ValidationResult(is_valid, reason=reason)


math_requirement = Requirement(
    description="Invoice arithmetic must be internally consistent.",
    validation_fn=validate_invoice_requirement,
    check_only=True,
)

In [ ]:
def extract_invoice_validated(email_text: str) -> Invoice:
    result = m.instruct(
        "Extract invoice information from: {{email}}",
        user_variables={"email": email_text},
        format=Invoice,
        requirements=[math_requirement],
        model_options={ModelOption.SEED: 42},
    )
    return Invoice.model_validate_json(result.value)

Run the validated Mellea path.

In [ ]:
print("MELLEA WITH VALIDATION")
validated_result = extract_invoice_validated(sample_email)

print(f"\n✓ Extracted and validated")
print(f"Subtotal: ${validated_result.subtotal}")
print(f"Tax: ${validated_result.tax}")
print(f"Total: ${validated_result.total}")
print("\n✓ All calculations verified")

#### Step 6: Compare Behavior on Ambiguous Input

Use a less precise email to see how each approach behaves when the source text is incomplete.

In [ ]:
print("TESTING WITH AMBIGUOUS INPUT")

print("\nRaw LLM:")
raw_prob = extract_invoice_raw(problematic_email)
if raw_prob:
    print(f"  Got result of type: {type(raw_prob)}")
    print("  Reliability still depends on manual validation.")
else:
    print("  Failed")

print("\nMellea:")
try:
    mellea_prob = extract_invoice_validated(problematic_email)
    print(f"  ✓ Extracted typed Invoice")
    print(f"  Invoice: {mellea_prob.invoice_number}")
    print(f"  Total: ${mellea_prob.total}")
except Exception as error:
    print(f"  Handled with explicit failure: {error}")

### Output Comparison

##### Clean Invoice (`sample_email`)

| | Raw LLM | Mellea |
|---|---|---|
| **Return type** | `dict` | `Invoice` (Pydantic model) |
| **Field access** | `result.get('invoice_number')` — no guarantee key exists | `result.invoice_number` — IDE autocomplete, type-checked |
| **Invoice number** | `INV-2024-0542` | `INV-2024-0542` |
| **Total** | `$415.8` | `$415.8` |
| **Items** | List of plain dicts | 3 typed `InvoiceItem` objects |
| **Math validation** | Not checked — your responsibility | `math_requirement` confirmed subtotal $385.0 + tax $30.8 = $415.8 ✓ |
| **Parse failures** | Silent `None` return | `ValidationError` with field-level detail |

##### Ambiguous Invoice (`problematic_email`)

| | Raw LLM | Mellea |
|---|---|---|
| **Return type** | `dict` | `Invoice` (Pydantic model) |
| **Invoice number** | Placeholder or hallucinated value | `"not provided"` — explicit about what was missing |
| **Total** | Best-guess value, no flag | `$415.0` — extracted from "around $415" |
| **Missing fields** | Silently filled or omitted | Schema forces every field to be accounted for |
| **Math check** | Never runs | Ran — flagged inconsistency between items and total |

---

#### Why Choose Mellea

| Concern | Raw LLM | Mellea |
|---|---|---|
| **Type safety** | `dict` — wrong key crashes at runtime | Pydantic model — wrong field fails at parse time with a clear message |
| **Validation** | Write it yourself every time | Attach a `Requirement`; Mellea runs it and can auto-repair |
| **Downstream code** | Guards everywhere (`if result and 'total' in result`) | Trust the object — `invoice.total` always exists and is a `float` |
| **Ambiguous input** | Silent partial data, hard to detect | Schema surfaces gaps explicitly; validation flags arithmetic errors |
| **Repair loop** | Manual retry logic | Built-in `RejectionSamplingStrategy` retries with the failure reason |
| **Observability** | Raw string in, dict out | Structured object with generation metadata attached |

### Key Takeaways

- Raw LLM calls return text that still needs parsing and validation
- Mellea returns typed objects that are easier to use in application code
- Custom requirements let you enforce business rules such as invoice math
- Validation and repair reduce downstream failures on structured extraction tasks
- Local Ollama models make this recipe runnable without watsonx credentials

### Next Steps

- [Instruct-Validate-Repair](../InstructValidateRepair/InstructValidateRepair.ipynb) - Deep dive into =
- [Quick Start](../QuickStart/QuickStart.ipynb) - Mellea basics